# 02 - Model Training & Validation Orchestration

**Competition:** Solar Filament Segmentation Challenge 2026  
**Stage:** Baseline Training (U-Net with ResNet34 backbone)

> **Principle:** Kaggle notebooks import modular Python components from `filament_seg`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

from filament_seg.utils import set_seed, load_config, get_device, get_logger
from filament_seg.model import build_segmentation_model
from filament_seg.losses import BCEDiceLoss
from filament_seg.dataset import SolarFilamentDataset
from filament_seg.preprocessing import ForegroundAwareCrop, get_train_transform, get_val_transform

project_root = Path.cwd().resolve()
assert (project_root / "src" / "filament_seg" / "dataset.py").is_file(), project_root
logger = get_logger("training_notebook")
config = load_config(project_root / "configs" / "baseline.yaml")
set_seed(config["project"]["seed"])
device = get_device()
print(f"Python: {sys.executable}")
print(f"Project: {project_root}")
print(f"Device selected: {device}")
print(f"Experiment: {config['project']['experiment_name']}")

Python: C:\Users\Vinny\Project\Kaggle\Solar Filament Segmentation\.conda\python.exe
Project: C:\Users\Vinny\Project\Kaggle\Solar Filament Segmentation
Device selected: cpu
Experiment: baseline-unet-resnet34


## Dataset and image-level split

Split physical filenames once so alternative annotation sets cannot cross validation boundaries.

In [2]:
train_dir = project_root / "data" / "filament-segmentation-2026" / "MAGFiLO_1.0_Kaggle_2026" / "train"
image_dir = train_dir / "train_images"
annotation_path = train_dir / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

train_dataset = SolarFilamentDataset(image_dir, annotation_path, transform=get_train_transform(512))
val_dataset = SolarFilamentDataset(image_dir, annotation_path, transform=get_val_transform(512))
assert len(train_dataset) == len(val_dataset) == 707
assert len(train_dataset.images_by_filename) == 707
assert sum(map(len, train_dataset.images_by_filename.values())) == 1154
assert any(len(sets) > 1 for sets in train_dataset.images_by_filename.values())

indices = np.random.default_rng(config["project"]["seed"]).permutation(len(train_dataset))
train_indices = indices[:565].tolist()
val_indices = indices[565:].tolist()
train_names = {train_dataset.filenames[i] for i in train_indices}
val_names = {val_dataset.filenames[i] for i in val_indices}
assert len(train_names) == 565 and len(val_names) == 142
assert train_names.isdisjoint(val_names)
train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)
print(f"Physical images: {len(train_dataset)}; train: {len(train_subset)}; validation: {len(val_subset)}")

Physical images: 707; train: 565; validation: 142


## Preprocessing and DataLoaders

In [3]:
# A synthetic foreground patch checks spatial alignment through cropping and augmentation.
probe = np.zeros((768, 768), dtype=np.uint8)
probe[330:430, 350:460] = 1
crop = ForegroundAwareCrop(patch_size=512, foreground_prob=1.0)
cropped = crop(image=probe.copy(), mask=probe.copy())
assert cropped["image"].shape == cropped["mask"].shape == (512, 512)
assert np.array_equal(cropped["image"], cropped["mask"])
assert cropped["mask"].sum() > 0
augmented = get_train_transform(512)(image=probe.copy(), mask=probe.copy())
assert augmented["image"].shape == augmented["mask"].shape == (512, 512)
assert np.array_equal(augmented["image"], augmented["mask"])
assert set(np.unique(augmented["mask"])) <= {0, 1}
validation_transform = get_val_transform(512)
first = validation_transform(image=probe.copy(), mask=probe.copy())
second = validation_transform(image=probe.copy(), mask=probe.copy())
assert np.array_equal(first["image"], second["image"])
assert np.array_equal(first["mask"], second["mask"])
print("Foreground crop, paired augmentation, binary labels, and deterministic validation: PASS")

Foreground crop, paired augmentation, binary labels, and deterministic validation: PASS


In [4]:
train_loader = DataLoader(train_subset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=4, shuffle=False, num_workers=0)
assert len(train_loader) == 142
assert len(val_loader) == 36

for name, loader in (("train", train_loader), ("validation", val_loader)):
    batch = next(iter(loader))
    assert set(batch) == {"image", "mask", "filename", "image_id"}
    assert batch["image"].shape == batch["mask"].shape == (4, 1, 512, 512)
    assert batch["image"].dtype == batch["mask"].dtype == torch.float32
    assert torch.isfinite(batch["image"]).all() and torch.isfinite(batch["mask"]).all()
    assert 0 <= batch["image"].min() and batch["image"].max() <= 1
    assert set(torch.unique(batch["mask"]).tolist()) <= {0.0, 1.0}
    assert len(batch["filename"]) == len(batch["image_id"]) == 4
    print(f"{name}: {len(loader)} batches, shape {tuple(batch['image'].shape)}, "
          f"image range [{batch['image'].min().item():.6f}, {batch['image'].max().item():.6f}], "
          f"mask values {torch.unique(batch['mask']).tolist()}")

train: 142 batches, shape (4, 1, 512, 512), image range [0.000000, 0.898039], mask values [0.0, 1.0]


validation: 36 batches, shape (4, 1, 512, 512), image range [0.286275, 0.737255], mask values [0.0, 1.0]


## Training Execution

Full training loop will be executed using `filament_seg.train.train_one_epoch` and `validate_one_epoch` once dataset schema is finalized.